# TRL: RLHF / PPO / DPO

A refresher on **TRL** (Transformer Reinforcement Learning), Hugging Face's library for the
**alignment / preference-tuning** stage that turns a fluent-but-unaligned base model into one that
follows instructions and matches human preferences. TRL bundles the post-SFT toolkit: **PPO** (the
classic RLHF loop with a reward model), **DPO** (the popular reward-model-free shortcut), and the
newer reward-style losses (KTO, ORPO, GRPO). The unifying idea is small: **push the model toward
responses humans prefer while staying anchored to where it started** — the `β`/KL knob you'll see in
every example below.

## 1. What & Why

A pretrained LLM predicts the next token; an SFT (supervised fine-tuned) model imitates good
demonstrations. Neither is trained to know that *between two plausible answers, humans prefer this
one*. **Preference tuning** closes that gap, and it's the stage that produced the jump from GPT-3 to
InstructGPT/ChatGPT.

The pipeline is three stages:

1. **SFT** — fine-tune on instruction→response demonstrations. Gets the format and basic helpfulness.
2. **Reward modeling** — train a model to score a response, fit on human **pairwise** judgments
   (*"A is better than B"*). The reward model is a learned stand-in for a human rater.
3. **RL / preference optimization** — update the policy to maximize reward **without drifting too far**
   from the SFT model (a KL penalty). This is where PPO/DPO live.

**Why a KL leash?** Pure reward maximization "reward-hacks": the model finds degenerate, high-scoring
gibberish the reward model over-rates, and forgets its language ability. Penalizing
`KL(policy ‖ reference)` keeps it fluent and on-distribution. That `β` knob — *how far am I allowed to
move?* — is the single most important hyperparameter in this whole family.

**Reach for TRL when** you have a base/SFT model and either (a) a reward signal or (b) a dataset of
chosen/rejected pairs, and you want to align tone, helpfulness, safety, or task behavior. **Don't**
reach for it to teach the model *new facts or formats* — that's SFT's job; alignment only reshapes
preferences over what the model can already produce.

## 2. Mental Model

Think of the SFT model as a **river already flowing in roughly the right direction**. Preference
tuning doesn't dig a new channel — it nudges the banks so the water pools where humans like it.

- **PPO** is the *interactive* version: the model generates a sample, a **reward model** grades it,
  and you take a small gradient step that raises the probability of high-reward samples — but a
  spring (the KL penalty) pulls you back toward the reference whenever you stray. Generate → score →
  nudge → repeat. Powerful, but it's online RL: four models in memory (policy, reference, reward,
  value head) and finicky to stabilize.

- **DPO** is the *algebraic shortcut*. Someone proved that the PPO objective has a **closed-form
  optimal policy**, `π*(y) ∝ π_ref(y)·exp(r(y)/β)`. Invert that and the reward model *disappears* —
  you can optimize directly on the preference pairs with a simple classification-style loss. No
  sampling, no reward model, no value head. One model, one supervised-looking loss.

The thing to hold onto: **PPO and DPO optimize the same objective** (KL-regularized reward
maximization). PPO does it the hard online way with an explicit reward model; DPO does it offline with
a clever loss. The `β` is the same spring constant in both.

## 3. Key Concepts

| Term | What it means |
|------|---------------|
| **Policy** | The model being trained — `π_θ`. Starts as a copy of the SFT model. |
| **Reference model** | A *frozen* copy of the SFT model, `π_ref`. The KL anchor; you measure drift against it. |
| **Reward model (RM)** | A model that maps a (prompt, response) to a scalar score, trained on human pairwise preferences (Bradley–Terry loss). Used by PPO; *not* needed by DPO. |
| **KL penalty / `β`** | Controls how far the policy may move from the reference. Low `β` → big reward gains but reward-hacking/collapse; high `β` → safe but barely changes. |
| **PPO** | Proximal Policy Optimization. Online RL: clipped policy-gradient updates from sampled generations scored by the RM, plus the KL term. |
| **DPO** | Direct Preference Optimization. Reward-model-free; a logistic loss on chosen vs rejected log-prob ratios against the reference. |
| **Implicit reward** | In DPO, `β·log(π_θ(y)/π_ref(y))` *is* the model's implicit reward for a response — DPO trains it to rank chosen above rejected. |
| **chosen / rejected** | The two halves of a preference pair: the response humans preferred vs the one they didn't. DPO/KTO/ORPO data. |
| **KTO / ORPO / GRPO** | Variants: **KTO** needs only thumbs-up/down (not pairs); **ORPO** folds preference into SFT (no reference model); **GRPO** is PPO-without-a-value-model, used by DeepSeek-R1-style reasoning RL. |

The **DPO loss** for a pair `(y_w` chosen, `y_l` rejected) given prompt `x`:

$$\mathcal{L}_{\text{DPO}} = -\log\sigma\!\Big(\beta\big[\big(\log\pi_\theta(y_w\,|\,x)-\log\pi_{\text{ref}}(y_w\,|\,x)\big) - \big(\log\pi_\theta(y_l\,|\,x)-\log\pi_{\text{ref}}(y_l\,|\,x)\big)\big]\Big)$$

Read it as: *make the chosen response more likely than the rejected one — measured relative to the
reference — and `σ` saturates once the margin is comfortably positive.*

## 4. Setup

Real preference tuning needs a GPU, a base model, and `trl` + `transformers` + `peft` + `datasets`.
But the **ideas** — the DPO loss and the KL-regularized RLHF objective — are tiny and run on CPU with
nothing but NumPy and PyTorch. The worked examples below implement both *from scratch* so the
understanding is in front of you; the real `DPOTrainer`/`PPOTrainer` API is shown in a **gated** cell
that only fires when you set an env var on a real GPU box.

```bash
# The real stack (needs a CUDA GPU for anything non-trivial):
pip install "trl>=0.12" transformers peft datasets accelerate bitsandbytes
```

In [ ]:
# Environment probe — what's available in THIS kernel (no GPU, no downloads needed).
import importlib.util
import sys

import numpy as np


def have(mod: str) -> str:
    return "installed" if importlib.util.find_spec(mod) else "not installed"


print(f"python        : {sys.version.split()[0]}")
print(f"numpy         : {np.__version__}")
for m in ("torch", "transformers", "trl", "peft", "datasets"):
    print(f"{m:<14}: {have(m)}")

## 5. Worked Examples

### Example 1 — the DPO loss, from scratch

DPO needs no reward model and no sampling: just chosen/rejected responses, the policy, and a frozen
reference. We model a toy "policy" as logits over four candidate responses to one prompt, start it
**equal to the reference**, and train it with the DPO loss on a handful of preference pairs. Watch
the loss fall, the probabilities reorder to respect the preferences, and the **implicit reward**
`β·log(π/π_ref)` go positive for the response humans like.

In [ ]:
import torch
import torch.nn.functional as F

torch.manual_seed(0)

responses = ["polite+correct", "polite+wrong", "rude+correct", "off-topic"]

# Reference = frozen SFT model. Here: uniform preference over the four responses.
ref_logits = torch.zeros(len(responses))

# Policy starts as an exact copy of the reference (the standard DPO init).
policy_logits = torch.nn.Parameter(ref_logits.clone())

# Human preference pairs as (chosen_idx, rejected_idx).
# Reading: response 0 beats everything; 2 beats 3.
pairs = [(0, 1), (0, 2), (0, 3), (2, 3)]

beta = 0.1
opt = torch.optim.SGD([policy_logits], lr=0.5)


def logprobs(logits):
    return F.log_softmax(logits, dim=-1)


def dpo_loss(p_logp, r_logp, chosen, rejected, beta):
    # (logπθ(yw) - logπref(yw)) - (logπθ(yl) - logπref(yl))
    margin = (p_logp[chosen] - r_logp[chosen]) - (p_logp[rejected] - r_logp[rejected])
    return -F.logsigmoid(beta * margin)


for step in range(81):
    p_logp, r_logp = logprobs(policy_logits), logprobs(ref_logits)
    loss = sum(dpo_loss(p_logp, r_logp, c, r, beta) for c, r in pairs) / len(pairs)
    opt.zero_grad()
    loss.backward()
    opt.step()
    if step % 20 == 0:
        print(f"step {step:>2}  loss {loss.item():.4f}")

p_logp, r_logp = logprobs(policy_logits), logprobs(ref_logits)
implicit_reward = (beta * (p_logp - r_logp)).tolist()

print("\n              ", "  ".join(f"{r:>14}" for r in responses))
print("ref   prob    ", "  ".join(f"{x:>14.3f}" for x in F.softmax(ref_logits, 0).tolist()))
print("dpo   prob    ", "  ".join(f"{x:>14.3f}" for x in F.softmax(policy_logits, 0).tolist()))
print("implicit reward", " ".join(f"{x:>14.3f}" for x in implicit_reward))

The policy moved exactly as the preferences dictate: `polite+correct` (preferred in every pair) gets
the most probability and the only positive implicit reward; `rude+correct` (which only beats
`off-topic`) comes second; the responses that lose their pairs are pushed down. No reward model was
ever trained — the preference pairs *are* the signal.

### Example 2 — the RLHF objective and the `β` knob (what PPO is solving)

PPO maximizes `E[r(y)] − β·KL(π ‖ π_ref)`. That objective has a **closed-form optimum**:
`π*(y) ∝ π_ref(y)·exp(r(y)/β)`. PPO reaches it by noisy online sampling; here we just compute it
directly to *see* what `β` controls. (This same closed form is what DPO inverts to eliminate the
reward model.) We sweep `β` and watch the trade-off: small `β` chases reward but drifts far from the
reference (and would reward-hack in the real world); large `β` stays safe but timid.

In [ ]:
# A reward model's scores for each response, and the reference (post-SFT) policy.
reward = np.array([2.0, -0.5, 0.5, -1.0])      # polite+correct best, off-topic worst
ref = np.array([0.40, 0.30, 0.20, 0.10])       # reference policy over the 4 responses

print(f"reference         E[r] = {ref @ reward:+.3f}   (starting point)\n")
print(f"{'beta':>6} | {'E[r]':>7} | {'KL(pi*||ref)':>12} | optimal policy pi*")
print("-" * 64)
for beta in (2.0, 0.5, 0.1):
    w = ref * np.exp(reward / beta)            # pi*(y) proportional to pi_ref * exp(r/beta)
    pi = w / w.sum()
    kl = float(np.sum(pi * np.log(pi / ref)))
    probs = "  ".join(f"{x:.3f}" for x in pi)
    print(f"{beta:>6} | {pi @ reward:>+7.3f} | {kl:>12.3f} | {probs}")

print("\nresponses:", responses)

Read the trade-off down the table: as `β` shrinks, expected reward climbs toward the maximum but the
KL from the reference explodes and the policy collapses onto the single highest-reward response — the
mathematical face of **reward hacking / mode collapse**. As `β` grows, the policy stays close to the
reference and barely improves. Tuning `β` (PPO's `kl_coef`, DPO's `beta`) is choosing *how aggressive*
to be, and it's the same knob in both algorithms.

### Example 3 — the real TRL `DPOTrainer` (gated)

In practice you never hand-roll the loss — you hand `trl` a preference dataset and a model. The cell
below is the **real, current API shape**, gated behind `RUN_TRL` + a GPU check so the notebook still
runs top-to-bottom on a plain CPU. Flip the env var on a GPU box with the libraries installed to
actually train. The `PPOConfig`/`PPOTrainer` path is sketched in the comment for contrast.

In [ ]:
import os

GPU = importlib.util.find_spec("torch") and __import__("torch").cuda.is_available()
HAVE_LIBS = all(importlib.util.find_spec(m) for m in ("trl", "transformers", "datasets"))

if os.getenv("RUN_TRL") and GPU and HAVE_LIBS:
    from datasets import Dataset
    from transformers import AutoModelForCausalLM, AutoTokenizer
    from trl import DPOConfig, DPOTrainer

    model_id = "Qwen/Qwen2.5-0.5B-Instruct"  # tiny instruct model = the SFT/reference start
    tok = AutoTokenizer.from_pretrained(model_id)
    model = AutoModelForCausalLM.from_pretrained(model_id)

    # Preference data: each row needs prompt / chosen / rejected text.
    data = Dataset.from_dict({
        "prompt":   ["Explain gravity to a 5-year-old.", "Write a polite refusal."],
        "chosen":   ["Things fall because Earth gently pulls them down.",
                     "I'm sorry, I can't help with that, but here's an alternative."],
        "rejected": ["Gravitation is the curvature of the stress-energy tensor.",
                     "No. Go away."],
    })

    cfg = DPOConfig(
        beta=0.1,                       # the KL knob from Example 2
        output_dir="dpo-demo",
        per_device_train_batch_size=1,
        max_steps=5,
        learning_rate=5e-6,
    )
    # ref_model=None -> DPOTrainer freezes a copy of `model` as the reference automatically.
    trainer = DPOTrainer(model=model, ref_model=None, args=cfg,
                         train_dataset=data, processing_class=tok)
    trainer.train()
    print("DPO training step complete.")

    # PPO instead would look like:
    #   from trl import PPOConfig, PPOTrainer
    #   trainer = PPOTrainer(PPOConfig(...), model, ref_model, reward_model, tok, dataset)
    #   # loop: query -> model.generate -> reward_model score -> trainer.step(queries, resp, rewards)
else:
    print("Gated: set RUN_TRL=1 on a GPU with `trl` installed to run a real DPO step.")
    print(f"  GPU available : {bool(GPU)}")
    print(f"  trl installed : {importlib.util.find_spec('trl') is not None}")

## 6. Gotchas & Pitfalls

- **DPO without an SFT model first usually underperforms.** DPO reshapes preferences over what the
  model already produces; if the base can't generate decent responses, there's nothing good to prefer.
  Run SFT (even a light pass on the chosen responses) **before** DPO.
- **`β` is everything.** Too low → the policy drifts, the implicit reward inflates, and you get the
  point-mass collapse from Example 2 (and degenerate, repetitive text). Too high → nothing changes.
  Typical DPO `β` is **0.1–0.5**; sweep it.
- **The reference must be frozen and correct.** A common bug is letting the reference track the policy
  (or pointing it at the wrong checkpoint). The KL anchor is meaningless if `π_ref` moves. With LoRA,
  the reference is the base model with adapters *disabled*, so `ref_model=None` is correct.
- **Chosen/rejected length bias.** DPO can learn "longer = better" (or shorter) if your preferred
  responses are systematically longer. Length-normalize or use a length-penalized variant if you see
  the model padding its answers.
- **PPO is hard to stabilize.** Four models in memory, reward normalization, KL-coefficient schedules,
  value-head warmup, generation length — many knobs, easy to diverge. This is *why* DPO got popular:
  comparable results, far fewer moving parts. Try DPO first; reach for PPO/GRPO when you need true
  online exploration (e.g., RL for reasoning with a verifiable reward).
- **Reward over-optimization is real even with DPO.** A higher implicit reward / preference accuracy
  on your pairs does **not** monotonically mean a better model — evaluate on held-out generations, not
  just the training loss.

## 7. When to Use vs Alternatives

| Method | Needs | Models in memory | Pros | Cons |
|--------|-------|------------------|------|------|
| **SFT** | demonstrations | 1 | Teaches format/skills; prerequisite for the rest | Imitates; can't express "A > B" |
| **PPO (RLHF)** | reward model + prompts | 4 (policy, ref, reward, value) | True online RL; can exceed the data; standard for classic RLHF | Complex, unstable, expensive |
| **DPO** | chosen/rejected pairs | 2 (policy, ref) | Simple, stable, no RM/sampling; strong default | Offline (no exploration); sensitive to `β` and data quality |
| **KTO** | thumbs up/down labels | 2 | No pairs needed — cheaper to label | Slightly weaker than good pairwise DPO |
| **ORPO** | chosen/rejected pairs | 1 (no reference!) | Folds preference into SFT in one stage | Newer; less battle-tested |
| **GRPO** | reward (often verifiable) | 2–3 (no value model) | PPO-style online RL, cheaper; powers reasoning-RL (DeepSeek-R1) | Still online RL complexity; needs a good reward |

**Rule of thumb:** Start with **SFT → DPO** — it's the cheapest path to a well-aligned model and the
strong default in 2024–2025. Use **KTO** if you can only get binary feedback, **ORPO** to collapse SFT
and alignment into one stage on a tight budget, and **PPO/GRPO** when you need genuine online
exploration against a reward signal (notably RL for reasoning with verifiable rewards). All of these
live in TRL behind near-identical `*Trainer` APIs, so switching is mostly a config change.

## 8. Resources

- **TRL documentation** — trainers, configs, and end-to-end examples for SFT/DPO/PPO/KTO/ORPO/GRPO:
  <https://huggingface.co/docs/trl>
- **DPO paper** — *Rafailov et al., "Direct Preference Optimization: Your Language Model is Secretly a
  Reward Model"* (NeurIPS 2023), the derivation behind Examples 1–2: <https://arxiv.org/abs/2305.18290>
- **InstructGPT paper** — *Ouyang et al., "Training language models to follow instructions with human
  feedback"*, the canonical three-stage RLHF/PPO pipeline: <https://arxiv.org/abs/2203.02155>
- **Hugging Face RLHF blog** — *"Illustrating Reinforcement Learning from Human Feedback (RLHF)"*, the
  best visual intro to reward models and the PPO loop: <https://huggingface.co/blog/rlhf>
- **TRL DPOTrainer guide** — dataset format, `beta`, and LoRA-with-DPO specifics:
  <https://huggingface.co/docs/trl/dpo_trainer>